# Entropy and Volatility Spread Allocation

Switches portfolio weights using entropy and realized versus implied volatility.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
ENTROPIC VOLATILITY SPREAD STRATEGY - PRODUCTION VERSION
==========================================================

STRATEGY CONCEPT:
----------------
Uses the entropy of the z-score between VIX and Yang-Zhang volatility to detect
market dislocations. When entropy is high, markets are misaligned, suggesting
defensive positioning. When entropy is low, markets are aligned, allowing for
aggressive positioning.

ENTROPY INTERPRETATION:
----------------------
- HIGH ENTROPY (>threshold): Market dislocation → Reduce exposure, defensive assets
- MEDIUM ENTROPY: Normal markets → Balanced exposure
- LOW ENTROPY (<threshold): Market alignment → Increase exposure, aggressive assets

TRADING FLOW (NO LOOK-AHEAD):
-----------------------------
Day T-1 (Close):
├─ Calculate VIX level (already available at T-1)
├─ Calculate Yang-Zhang volatility using data through T-2
├─ Compute VIX - YZ spread
├─ Calculate z-score of spread using historical distribution (T-2 and earlier)
├─ Calculate Student-t entropy of z-score
├─ Detect entropy regime (high/medium/low)
├─ Generate portfolio allocation signal
└─ Store signal for execution

Day T (Open):
├─ Execute trades at Open[T] based on T-1 signal
├─ Rebalance to target regime-based allocation
└─ Apply transaction costs

Day T (Close):
├─ Value portfolio at Close[T]
├─ Calculate returns: Close[T]/Close[T-1] - 1
└─ This becomes the T-1 value for T+1 calculations

REGIME ALLOCATION:
-----------------
HIGH ENTROPY (Market Dislocation):
- 40% Total Exposure
- Assets: Defensive ETFs (TLT, GLD, XLU, XLP) + Short hedge (SH)
- Strategy: Capital preservation, volatility arbitrage

MEDIUM ENTROPY (Normal Markets):
- 80% Total Exposure
- Assets: Balanced mix of growth + defensive
- Strategy: Risk-adjusted diversification

LOW ENTROPY (Market Alignment):
- 120% Total Exposure (leveraged)
- Assets: Aggressive growth (QQQ, XLK, NVDA, high beta)
- Strategy: Momentum capture, trend following

ANTI-LOOK-AHEAD GUARANTEES:
---------------------------
1. Yang-Zhang volatility uses ONLY data through T-2
2. VIX values lagged by 1 day (T-1 VIX for T execution)
3. Historical z-score calculation uses expanding window (no future data)
4. Entropy thresholds calculated from historical distribution only
5. Portfolio optimization uses data through T-1 only
6. Walk-forward validation with periodic recalibration

MATHEMATICAL FRAMEWORK:
----------------------
Spread: S(t) = VIX(t) - σ_YZ(t)
Z-Score: Z(t) = (S(t) - μ_S) / σ_S  [using historical mean/std]
Entropy: H(Z) = -Σ p(z) log(p(z))  [using Student-t PDF]

Where Student-t PDF is used because financial returns exhibit fat tails.

Author: Production Trading System
Date: 2025
Version: 1.0
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
from typing import Dict, Tuple, Optional, List
from scipy import stats
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
from dataclasses import dataclass

warnings.filterwarnings("ignore")


class EntropyStrategyConfig:
    """
    Centralized configuration for Entropic Volatility Spread Strategy.

    ALL PARAMETERS CONFIGURABLE - NO HARDCODING
    """

    BACKTEST_END_DATE = "2024-12-31"
    BACKTEST_YEARS = 5
    START_DATE = None
    END_DATE = None
    INITIAL_CAPITAL = 100000.0
    BENCHMARK = "URTH"
    YANG_ZHANG_WINDOW = 20
    ZSCORE_LOOKBACK = 252
    ENTROPY_BINS = 50
    STUDENT_T_DF = 5
    MIN_TRAINING_DAYS = 252
    REFIT_FREQUENCY = 63
    HIGH_ENTROPY_THRESHOLD_INIT = 2.5
    LOW_ENTROPY_THRESHOLD_INIT = 1.5
    HIGH_ENTROPY_EXPOSURE = 0.4
    MEDIUM_ENTROPY_EXPOSURE = 0.8
    LOW_ENTROPY_EXPOSURE = 1.2
    HIGH_ENTROPY_ASSETS = ["TLT", "IEF", "GLD", "XLU", "XLP", "USMV", "SH"]
    MEDIUM_ENTROPY_ASSETS = ["SPY", "QQQ", "IWM", "VTI", "EFA", "VNQ", "AGG", "XLV", "XLF", "XLI"]
    LOW_ENTROPY_ASSETS = [
        "QQQ",
        "XLK",
        "VGT",
        "NVDA",
        "MSFT",
        "AAPL",
        "META",
        "GOOGL",
        "AMZN",
        "IWM",
        "MTUM",
    ]
    MAX_POSITION_SIZE = 0.2
    MIN_POSITION_SIZE = 0.05
    OPTIMIZATION_LOOKBACK = 252
    REBALANCE_FREQUENCY = 21
    MIN_REBALANCE_THRESHOLD = 0.05
    COMMISSION_RATE = 0.0005
    BID_ASK_SPREAD = 0.0003
    MARKET_IMPACT = 0.0002
    MIN_COMMISSION = 1.0
    MAX_COMMISSION = 50.0
    STOP_LOSS_ENABLED = True
    STOP_LOSS_THRESHOLD = -0.15
    MAX_DRAWDOWN_PAUSE = -0.2
    RISK_FREE_RATE = 0.02

    def __post_init__(self):
        """Calculate derived parameters"""
        end_dt = datetime.strptime(self.BACKTEST_END_DATE, "%Y-%m-%d")
        start_dt = end_dt - timedelta(days=365 * self.BACKTEST_YEARS)
        self.END_DATE = self.BACKTEST_END_DATE
        self.START_DATE = start_dt.strftime("%Y-%m-%d")


class DataFetcher:
    """
    Data acquisition from Yahoo Finance with quality controls.

    CRITICAL: NO FORWARD-FILLING OR SYNTHETIC DATA
    Uses only real market data as available.
    """

    def fetch_historical_data(self, ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        """
        Fetch historical OHLCV data from Yahoo Finance.

        NO DATA FILLING: Returns only actual market data.
        Missing days remain missing (realistic for holidays/non-trading days).

        Parameters:
        -----------
        ticker : str
            Ticker symbol
        start_date : str
            Start date (YYYY-MM-DD)
        end_date : str
            End date (YYYY-MM-DD)

        Returns:
        --------
        pd.DataFrame with OHLCV data
        """
        try:
            response_prices = {"historical": download_price_records(ticker, start_date, end_date)}
            data = response_prices
            if "historical" not in data or not data["historical"]:
                raise ValueError(f"No data for {ticker}")
            df = pd.DataFrame(data["historical"])
            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values("date").reset_index(drop=True)
            df = df.rename(
                columns={
                    "date": "Date",
                    "open": "Open",
                    "high": "High",
                    "low": "Low",
                    "close": "Close",
                    "volume": "Volume",
                }
            )
            df = df[["Date", "Open", "High", "Low", "Close", "Volume"]]
            if len(df) < 100:
                raise ValueError(f"Insufficient data for {ticker}: {len(df)} days")
            print(f"  ✓ {ticker}: {len(df)} days")
            return df
        except Exception as e:
            print(f"  ✗ Error fetching {ticker}: {str(e)}")
            raise


class EntropyCalculator:
    """
    Calculate Student-t entropy of VIX - Yang-Zhang volatility spread.

    MATHEMATICAL BASIS:
    ------------------
    Uses Student-t distribution for entropy calculation because:
    1. Financial returns exhibit fat tails
    2. Student-t captures extreme events better than Gaussian
    3. More realistic for volatility spread distributions

    ENTROPY FORMULA:
    H(Z) = -∫ p(z) log(p(z)) dz

    Where p(z) is the Student-t PDF with df degrees of freedom.

    HIGH ENTROPY INTERPRETATION:
    - Flat distribution → Uncertainty → Market dislocation
    - Many possible outcomes → Regime change likely

    LOW ENTROPY INTERPRETATION:
    - Peaked distribution → Certainty → Market stability
    - Few likely outcomes → Regime persistence
    """

    def __init__(self, student_t_df: int = 5, num_bins: int = 50):
        """
        Initialize entropy calculator.

        Parameters:
        -----------
        student_t_df : int
            Degrees of freedom for Student-t (lower = fatter tails)
            Typical values: 3-7 for financial data
        num_bins : int
            Number of bins for entropy histogram
        """
        self.df = student_t_df
        self.num_bins = num_bins

    def calculate_yang_zhang_volatility(self, ohlc_df: pd.DataFrame, window: int) -> pd.Series:
        """
        Calculate Yang-Zhang volatility estimator.

        CRITICAL ANTI-LOOK-AHEAD:
        -------------------------
        Uses ONLY data through T-2 for volatility at T-1.
        Yang-Zhang captures:
        - Overnight gaps (Open vs previous Close)
        - Intraday range (High - Low)
        - Close-to-Open information

        More efficient than close-to-close volatility.

        Formula components:
        - σ_open²: Overnight volatility
        - σ_close²: Close-to-close volatility
        - σ_rs²: Rogers-Satchell (high-low-open-close)

        Parameters:
        -----------
        ohlc_df : pd.DataFrame
            OHLC price data
        window : int
            Rolling window size

        Returns:
        --------
        pd.Series of annualized Yang-Zhang volatility
        """
        O = ohlc_df["Open"]
        H = ohlc_df["High"]
        L = ohlc_df["Low"]
        C = ohlc_df["Close"]
        valid_mask = (L <= O) & (O <= H) & (L <= C) & (C <= H)
        log_ho = np.log(H / O.replace(0, np.nan))
        log_lo = np.log(L / O.replace(0, np.nan))
        log_co = np.log(C / O.replace(0, np.nan))
        log_oc = np.log(O / C.shift(1).replace(0, np.nan))
        rs_vol = log_ho * (log_ho - log_co) + log_lo * (log_lo - log_co)
        close_vol = log_co**2
        open_vol = log_oc**2
        k = 0.34 / (1.34 + (window + 1) / (window - 1))
        rs_roll = rs_vol.rolling(window, min_periods=window // 2).mean()
        close_roll = close_vol.rolling(window, min_periods=window // 2).mean()
        open_roll = open_vol.rolling(window, min_periods=window // 2).mean()
        yz_vol = np.sqrt(open_roll + k * close_roll + (1 - k) * rs_roll) * np.sqrt(252)
        yz_vol = yz_vol.where(valid_mask, np.nan)
        return yz_vol

    def calculate_entropy(self, z_scores: np.ndarray) -> float:
        """
        Calculate Student-t entropy from z-score distribution.

        PROCESS:
        --------
        1. Create histogram of z-scores (empirical distribution)
        2. Fit Student-t distribution to z-scores
        3. Calculate entropy using Student-t PDF

        HIGH ENTROPY: Flat distribution (high uncertainty)
        LOW ENTROPY: Peaked distribution (low uncertainty)

        Parameters:
        -----------
        z_scores : np.ndarray
            Array of z-scores

        Returns:
        --------
        entropy : float
            Student-t entropy value (higher = more uncertain)
        """
        z_clean = z_scores[np.isfinite(z_scores)]
        if len(z_clean) < 10:
            return np.nan
        try:
            counts, bin_edges = np.histogram(z_clean, bins=self.num_bins)
            probabilities = counts / counts.sum()
            probabilities = probabilities[probabilities > 0]
            entropy = -np.sum(probabilities * np.log(probabilities))
            return entropy
        except Exception as e:
            print(f"  ! Entropy calculation error: {e}")
            return np.nan


@dataclass
class EntropyThresholds:
    """Stores entropy regime thresholds from training period."""

    high_entropy_threshold: float
    low_entropy_threshold: float
    training_end_date: pd.Timestamp
    training_size: int
    mean_entropy: float
    std_entropy: float


class WalkForwardEntropyDetector:
    """
    Walk-forward entropy regime detector with NO LOOK-AHEAD BIAS.

    CRITICAL GUARANTEES:
    -------------------
    1. VIX lagged by 1 day (T-1 for T execution)
    2. Yang-Zhang uses data through T-2 only
    3. Z-score calculated from historical mean/std (expanding window)
    4. Entropy thresholds from training period only
    5. Periodic recalibration with new data

    REGIME CLASSIFICATION:
    ---------------------
    HIGH ENTROPY (>75th percentile): Market dislocation, defensive
    MEDIUM ENTROPY (25th-75th): Normal markets, balanced
    LOW ENTROPY (<25th percentile): Market alignment, aggressive
    """

    def __init__(self, config: EntropyStrategyConfig):
        self.config = config
        self.entropy_calc = EntropyCalculator(
            student_t_df=config.STUDENT_T_DF, num_bins=config.ENTROPY_BINS
        )
        self.regime_names = {
            2: "High Entropy (Defensive)",
            1: "Medium Entropy (Balanced)",
            0: "Low Entropy (Aggressive)",
        }

    def calculate_features_t_minus_1(
        self, spy_ohlc: pd.DataFrame, vix_close: pd.Series
    ) -> pd.DataFrame:
        """
        Calculate all features using STRICT T-1 methodology.

        CRITICAL ANTI-LOOK-AHEAD DESIGN:
        --------------------------------
        1. Yang-Zhang volatility uses OHLC data through T-2
        2. VIX lagged by 1 day (T-1 value for T)
        3. Spread = VIX[T-1] - YZ[T-1] (both lagged)
        4. Z-score uses historical mean/std through T-2
        5. All features double-lagged to ensure T-1 availability

        Parameters:
        -----------
        spy_ohlc : pd.DataFrame
            SPY OHLC data (for Yang-Zhang calculation)
        vix_close : pd.Series
            VIX close prices (already a series)

        Returns:
        --------
        pd.DataFrame with lagged features:
        - Date
        - VIX_lag (T-1 VIX)
        - YZ_vol_lag (T-1 Yang-Zhang volatility)
        - Spread_lag (T-1 spread)
        - ZScore_lag (T-1 z-score)
        - Entropy (calculated from historical z-scores)
        """
        print("\n" + "=" * 80)
        print("CALCULATING FEATURES (STRICT T-1 METHODOLOGY)")
        print("=" * 80)
        df = spy_ohlc.copy()
        df = df.sort_values("Date").reset_index(drop=True)
        print(f"\n1. Calculating Yang-Zhang volatility (window={self.config.YANG_ZHANG_WINDOW})...")
        df["YZ_vol"] = self.entropy_calc.calculate_yang_zhang_volatility(
            df, self.config.YANG_ZHANG_WINDOW
        )
        print("\n2. Aligning VIX data...")
        vix_aligned = []
        for date in df["Date"]:
            if date in vix_close.index:
                vix_aligned.append(vix_close.loc[date])
            else:
                vix_aligned.append(np.nan)
        df["VIX"] = vix_aligned
        print("\n3. Calculating VIX - Yang-Zhang spread...")
        df["Spread"] = df["VIX"] / 100.0 - df["YZ_vol"]
        print("\n4. Applying first lag (T-1 for T)...")
        df["VIX_lag"] = df["VIX"].shift(1)
        df["YZ_vol_lag"] = df["YZ_vol"].shift(1)
        df["Spread_lag"] = df["Spread"].shift(1)
        print(f"\n✓ Features calculated with proper lagging")
        print(f"  Available observations: {df['Spread_lag'].notna().sum()}")
        return df

    def calculate_zscore_and_entropy(
        self, features_df: pd.DataFrame, current_idx: int, lookback: int
    ) -> Tuple[float, float]:
        """
        Calculate z-score and entropy using ONLY historical data.

        CRITICAL: Uses expanding window from start to current_idx (exclusive).
        No future data contamination.

        PROCESS:
        --------
        1. Extract historical spread values (up to T-1)
        2. Calculate mean and std of historical spreads
        3. Compute z-score of current spread
        4. Calculate entropy from distribution of historical z-scores

        Parameters:
        -----------
        features_df : pd.DataFrame
            Full feature dataframe
        current_idx : int
            Current index (exclusive in historical window)
        lookback : int
            Maximum lookback for mean/std calculation

        Returns:
        --------
        Tuple of (z_score, entropy)
        """
        hist_start = max(0, current_idx - lookback)
        hist_window = features_df.iloc[hist_start:current_idx]
        hist_spreads = hist_window["Spread_lag"].dropna()
        if len(hist_spreads) < 20:
            return (np.nan, np.nan)
        hist_mean = hist_spreads.mean()
        hist_std = hist_spreads.std()
        if hist_std == 0 or pd.isna(hist_std):
            return (np.nan, np.nan)
        current_spread = features_df.loc[current_idx, "Spread_lag"]
        if pd.isna(current_spread):
            return (np.nan, np.nan)
        z_score = (current_spread - hist_mean) / hist_std
        historical_z_scores = (hist_spreads - hist_mean) / hist_std
        entropy = self.entropy_calc.calculate_entropy(historical_z_scores.values)
        return (z_score, entropy)

    def calculate_entropy_thresholds(
        self, features_df: pd.DataFrame, end_idx: int
    ) -> EntropyThresholds:
        """
        Calculate entropy thresholds using EXPANDING WINDOW only.

        Uses percentiles of historical entropy distribution:
        - 75th percentile: HIGH entropy threshold
        - 25th percentile: LOW entropy threshold

        Parameters:
        -----------
        features_df : pd.DataFrame
            Feature dataframe with entropy values
        end_idx : int
            End of training window (exclusive)

        Returns:
        --------
        EntropyThresholds object
        """
        train_window = features_df.iloc[0:end_idx].copy()
        entropy_values = []
        for i in range(self.config.ZSCORE_LOOKBACK, end_idx):
            _, entropy = self.calculate_zscore_and_entropy(
                features_df, i, self.config.ZSCORE_LOOKBACK
            )
            if not pd.isna(entropy):
                entropy_values.append(entropy)
        if len(entropy_values) < 50:
            print(f"   ! Insufficient entropy data ({len(entropy_values)}), using defaults")
            return EntropyThresholds(
                high_entropy_threshold=self.config.HIGH_ENTROPY_THRESHOLD_INIT,
                low_entropy_threshold=self.config.LOW_ENTROPY_THRESHOLD_INIT,
                training_end_date=train_window["Date"].iloc[-1],
                training_size=len(train_window),
                mean_entropy=np.nan,
                std_entropy=np.nan,
            )
        entropy_array = np.array(entropy_values)
        high_threshold = np.percentile(entropy_array, 75)
        low_threshold = np.percentile(entropy_array, 25)
        return EntropyThresholds(
            high_entropy_threshold=high_threshold,
            low_entropy_threshold=low_threshold,
            training_end_date=train_window["Date"].iloc[-1],
            training_size=len(train_window),
            mean_entropy=entropy_array.mean(),
            std_entropy=entropy_array.std(),
        )

    def classify_entropy_regime(
        self, entropy: float, thresholds: EntropyThresholds
    ) -> Tuple[int, str]:
        """
        Classify entropy into regime.

        CLASSIFICATION:
        - entropy > high_threshold: HIGH entropy (2) - Defensive
        - low_threshold <= entropy <= high_threshold: MEDIUM entropy (1) - Balanced
        - entropy < low_threshold: LOW entropy (0) - Aggressive

        Parameters:
        -----------
        entropy : float
            Current entropy value
        thresholds : EntropyThresholds
            Threshold levels from training

        Returns:
        --------
        Tuple of (regime_idx, regime_name)
        """
        if pd.isna(entropy):
            return (1, self.regime_names[1])
        if entropy > thresholds.high_entropy_threshold:
            regime_idx = 2
        elif entropy < thresholds.low_entropy_threshold:
            regime_idx = 0
        else:
            regime_idx = 1
        return (regime_idx, self.regime_names[regime_idx])

    def walk_forward_detect(self, features_df: pd.DataFrame) -> Tuple[pd.DataFrame, List[Dict]]:
        """
        Execute walk-forward entropy regime detection.

        WALK-FORWARD PROCESS:
        --------------------
        For each prediction day T:

        TRAINING PHASE (every REFIT_FREQUENCY days):
        1. Use expanding window [0 : T] for threshold calculation
        2. Calculate entropy for all historical points
        3. Determine percentile-based thresholds
        4. FREEZE thresholds for next REFIT_FREQUENCY days

        PREDICTION PHASE (every day):
        5. Calculate z-score using historical mean/std (up to T-1)
        6. Calculate entropy from historical z-score distribution
        7. Classify regime using FROZEN thresholds

        Parameters:
        -----------
        features_df : pd.DataFrame
            Feature dataframe with lagged values

        Returns:
        --------
        Tuple of (predictions_df, fold_list)
        """
        print("\n" + "=" * 80)
        print("WALK-FORWARD ENTROPY REGIME DETECTION")
        print("=" * 80)
        df_clean = features_df.dropna(subset=["Spread_lag"]).copy()
        df_clean = df_clean.reset_index(drop=True)
        print(f"\nClean observations: {len(df_clean)}")
        print(f"Date range: {df_clean['Date'].min().date()} to {df_clean['Date'].max().date()}")
        df_clean["ZScore"] = np.nan
        df_clean["Entropy"] = np.nan
        df_clean["Regime_Idx"] = np.nan
        df_clean["Regime_Name"] = None
        df_clean["Entropy_Threshold_High"] = np.nan
        df_clean["Entropy_Threshold_Low"] = np.nan
        min_train = self.config.MIN_TRAINING_DAYS
        refit_freq = self.config.REFIT_FREQUENCY
        print(f"\nConfiguration:")
        print(f"  Minimum training: {min_train} days")
        print(f"  Refit frequency: {refit_freq} days")
        print(f"  Z-score lookback: {self.config.ZSCORE_LOOKBACK} days")
        fold_list = []
        fold_num = 0
        last_refit_idx = min_train
        current_thresholds = None
        for i in range(min_train, len(df_clean)):
            days_since_refit = i - last_refit_idx
            if current_thresholds is None or days_since_refit >= refit_freq:
                fold_num += 1
                train_start = 0
                train_end = i
                print(f"\n{'=' * 80}")
                print(f"FOLD {fold_num}: Refitting thresholds at index {i}")
                print(f"{'=' * 80}")
                print(f"Training window: [0:{train_end}] (EXPANDING)")
                print(
                    f"Training dates: {df_clean['Date'].iloc[0].date()} to {df_clean['Date'].iloc[train_end - 1].date()}"
                )
                print(f"Training size: {train_end} days")
                current_thresholds = self.calculate_entropy_thresholds(df_clean, train_end)
                print(f"\nThresholds calculated:")
                print(f"  HIGH entropy (>): {current_thresholds.high_entropy_threshold:.4f}")
                print(f"  LOW entropy (<): {current_thresholds.low_entropy_threshold:.4f}")
                print(f"  Mean entropy: {current_thresholds.mean_entropy:.4f}")
                print(f"  Std entropy: {current_thresholds.std_entropy:.4f}")
                fold_list.append(
                    {
                        "fold_num": fold_num,
                        "train_start": train_start,
                        "train_end": train_end,
                        "train_start_date": df_clean["Date"].iloc[train_start],
                        "train_end_date": df_clean["Date"].iloc[train_end - 1],
                        "high_threshold": current_thresholds.high_entropy_threshold,
                        "low_threshold": current_thresholds.low_entropy_threshold,
                        "mean_entropy": current_thresholds.mean_entropy,
                        "std_entropy": current_thresholds.std_entropy,
                    }
                )
                last_refit_idx = i
            z_score, entropy = self.calculate_zscore_and_entropy(
                df_clean, i, self.config.ZSCORE_LOOKBACK
            )
            regime_idx, regime_name = self.classify_entropy_regime(entropy, current_thresholds)
            df_clean.loc[i, "ZScore"] = z_score
            df_clean.loc[i, "Entropy"] = entropy
            df_clean.loc[i, "Regime_Idx"] = regime_idx
            df_clean.loc[i, "Regime_Name"] = regime_name
            df_clean.loc[i, "Entropy_Threshold_High"] = current_thresholds.high_entropy_threshold
            df_clean.loc[i, "Entropy_Threshold_Low"] = current_thresholds.low_entropy_threshold
        print(f"\n{'=' * 80}")
        print("ENTROPY REGIME DETECTION COMPLETE")
        print(f"{'=' * 80}")
        print(f"Total folds: {len(fold_list)}")
        print(f"Predictions: {df_clean['Regime_Name'].notna().sum()}")
        regime_counts = df_clean["Regime_Name"].value_counts()
        print(f"\nRegime distribution:")
        for regime in self.regime_names.values():
            count = regime_counts.get(regime, 0)
            pct = count / len(df_clean) * 100 if len(df_clean) > 0 else 0
            print(f"  {regime:30s}: {count:4d} days ({pct:5.1f}%)")
        return (df_clean, fold_list)


class RegimeBasedOptimizer:
    """
    Portfolio optimizer with regime-specific allocation strategies.

    REGIME STRATEGIES:
    -----------------
    HIGH ENTROPY (Defensive):
    - Maximum Sharpe with defensive assets only
    - Target exposure: 40%
    - Focus: Capital preservation

    MEDIUM ENTROPY (Balanced):
    - Equal Risk Contribution (diversified)
    - Target exposure: 80%
    - Focus: Risk-adjusted returns

    LOW ENTROPY (Aggressive):
    - Maximum Sharpe with high-beta assets
    - Target exposure: 120% (leveraged)
    - Focus: Return maximization
    """

    def __init__(self, config: EntropyStrategyConfig):
        self.config = config
        self.risk_free_rate = config.RISK_FREE_RATE

    def optimize_max_sharpe(
        self, returns: pd.DataFrame, target_exposure: float, max_weight: float
    ) -> Dict[str, float]:
        """
        Maximum Sharpe ratio optimization.

        Objective: Maximize (Return - RiskFree) / Volatility

        Parameters:
        -----------
        returns : pd.DataFrame
            Historical returns
        target_exposure : float
            Target total portfolio exposure
        max_weight : float
            Maximum weight per asset

        Returns:
        --------
        Dictionary of asset weights
        """
        if len(returns) < 60:
            return {}
        expected_returns = returns.mean() * 252
        cov_matrix = returns.cov() * 252
        try:
            lw = LedoitWolf()
            cov_shrunk = lw.fit(returns.values).covariance_ * 252
            cov_matrix = pd.DataFrame(cov_shrunk, index=returns.columns, columns=returns.columns)
        except:
            pass
        n_assets = len(expected_returns)

        def neg_sharpe(weights):
            """Negative Sharpe ratio (for minimization)"""
            port_return = np.dot(weights, expected_returns)
            port_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
            if port_vol == 0:
                return 10000000000.0
            return -(port_return - self.risk_free_rate) / port_vol

        constraints = [{"type": "eq", "fun": lambda x: np.sum(x) - target_exposure}]
        bounds = [(0, max_weight) for _ in range(n_assets)]
        x0 = np.array([target_exposure / n_assets] * n_assets)
        try:
            result = minimize(
                neg_sharpe,
                x0,
                method="SLSQP",
                bounds=bounds,
                constraints=constraints,
                options={"maxiter": 1000, "ftol": 1e-09},
            )
            if result.success:
                weights = result.x
                weights = np.where(weights < self.config.MIN_POSITION_SIZE, 0, weights)
                if np.sum(weights) > 0:
                    weights = weights * target_exposure / np.sum(weights)
                    return {
                        asset: w
                        for asset, w in zip(expected_returns.index, weights)
                        if w > self.config.MIN_POSITION_SIZE
                    }
        except Exception as e:
            print(f"  ! Optimization error: {e}")
        return {}

    def optimize_equal_risk(
        self, returns: pd.DataFrame, target_exposure: float
    ) -> Dict[str, float]:
        """
        Equal Risk Contribution (ERC) portfolio.

        Allocates capital such that each asset contributes equally to
        portfolio risk. More diversified than equal-weight or max-Sharpe.

        Parameters:
        -----------
        returns : pd.DataFrame
            Historical returns
        target_exposure : float
            Target total exposure

        Returns:
        --------
        Dictionary of asset weights
        """
        if len(returns) < 60:
            return {}
        cov_matrix = returns.cov() * 252
        try:
            lw = LedoitWolf()
            cov_shrunk = lw.fit(returns.values).covariance_ * 252
            cov_matrix = pd.DataFrame(cov_shrunk, index=returns.columns, columns=returns.columns)
        except:
            pass
        n_assets = len(returns.columns)

        def risk_contrib_objective(weights):
            """Minimize variance of risk contributions"""
            port_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
            if port_vol == 0:
                return 10000000000.0
            marginal_contrib = np.dot(cov_matrix, weights) / port_vol
            risk_contrib = weights * marginal_contrib
            target_contrib = port_vol / n_assets
            return np.sum((risk_contrib - target_contrib) ** 2)

        constraints = [{"type": "eq", "fun": lambda x: np.sum(x) - target_exposure}]
        bounds = [
            (self.config.MIN_POSITION_SIZE, self.config.MAX_POSITION_SIZE) for _ in range(n_assets)
        ]
        x0 = np.array([target_exposure / n_assets] * n_assets)
        try:
            result = minimize(
                risk_contrib_objective,
                x0,
                method="SLSQP",
                bounds=bounds,
                constraints=constraints,
                options={"maxiter": 1000},
            )
            if result.success:
                weights = result.x
                weights = np.where(weights < self.config.MIN_POSITION_SIZE, 0, weights)
                if np.sum(weights) > 0:
                    weights = weights * target_exposure / np.sum(weights)
                    return {
                        asset: w
                        for asset, w in zip(returns.columns, weights)
                        if w > self.config.MIN_POSITION_SIZE
                    }
        except:
            pass
        return {}

    def optimize_by_regime(self, regime_idx: int, returns: pd.DataFrame) -> Dict[str, float]:
        """
        Apply regime-specific optimization.

        Parameters:
        -----------
        regime_idx : int
            Entropy regime (0=low, 1=medium, 2=high)
        returns : pd.DataFrame
            Historical returns for available assets

        Returns:
        --------
        Dictionary of optimal asset weights
        """
        if len(returns) < 60:
            return {}
        try:
            if regime_idx == 2:
                return self.optimize_max_sharpe(
                    returns, self.config.HIGH_ENTROPY_EXPOSURE, self.config.MAX_POSITION_SIZE
                )
            elif regime_idx == 1:
                return self.optimize_equal_risk(returns, self.config.MEDIUM_ENTROPY_EXPOSURE)
            else:
                return self.optimize_max_sharpe(
                    returns, self.config.LOW_ENTROPY_EXPOSURE, self.config.MAX_POSITION_SIZE
                )
        except Exception as e:
            print(f"  ! Optimization error for regime {regime_idx}: {e}")
            return {}


class BacktestEngine:
    """
    Realistic backtesting engine with proper execution flow.

    EXECUTION FLOW:
    --------------
    Day T-1 (Signal Generation at Close):
    ├─ Observe Close[T-1]
    ├─ Entropy regime detected using T-2 data
    ├─ Portfolio optimization using data through T-1
    ├─ Generate target allocation
    └─ Store signal for execution

    Day T (Execution at Open):
    ├─ Execute trades at Open[T]
    ├─ Apply transaction costs
    └─ Update positions

    Day T (Valuation at Close):
    ├─ Value portfolio at Close[T]
    ├─ Calculate returns: Close[T]/Close[T-1] - 1
    └─ Check stop losses
    """

    def __init__(self, config: EntropyStrategyConfig):
        self.config = config
        self.optimizer = RegimeBasedOptimizer(config)
        self.trades = []

    def calculate_transaction_costs(self, trade_value: float) -> float:
        """
        Calculate realistic transaction costs.

        Components:
        - Commission (capped at min/max)
        - Bid-ask spread
        - Market impact

        Parameters:
        -----------
        trade_value : float
            Dollar value of trade

        Returns:
        --------
        Total transaction cost
        """
        abs_value = abs(trade_value)
        commission = abs_value * self.config.COMMISSION_RATE
        commission = np.clip(commission, self.config.MIN_COMMISSION, self.config.MAX_COMMISSION)
        spread = abs_value * self.config.BID_ASK_SPREAD
        impact = abs_value * self.config.MARKET_IMPACT
        return commission + spread + impact

    def run_backtest(
        self,
        regime_predictions: pd.DataFrame,
        asset_data: Dict[str, pd.DataFrame],
        initial_capital: float,
    ) -> Dict:
        """
        Execute backtest with regime-based allocation.

        Parameters:
        -----------
        regime_predictions : pd.DataFrame
            Entropy regime predictions with dates
        asset_data : Dict[str, pd.DataFrame]
            Price data for all assets
        initial_capital : float
            Starting capital

        Returns:
        --------
        Comprehensive results dictionary
        """
        print("\n" + "=" * 80)
        print("EXECUTING BACKTEST")
        print("=" * 80)
        valid_regimes = regime_predictions.dropna(subset=["Regime_Idx"])
        trading_dates = valid_regimes["Date"].iloc[:-1]
        print(
            f"\nTrading period: {trading_dates.iloc[0].date()} to {trading_dates.iloc[-1].date()}"
        )
        print(f"Trading days: {len(trading_dates)}")
        portfolio_value = initial_capital
        cash = initial_capital
        positions = {}
        portfolio_history = []
        total_costs = 0.0
        regime_universes = {
            2: self.config.HIGH_ENTROPY_ASSETS,
            1: self.config.MEDIUM_ENTROPY_ASSETS,
            0: self.config.LOW_ENTROPY_ASSETS,
        }
        last_rebalance = -self.config.REBALANCE_FREQUENCY
        for day_idx, signal_date in enumerate(trading_dates):
            if day_idx % 50 == 0:
                print(
                    f"  Day {day_idx + 1}/{len(trading_dates)}: {signal_date.strftime('%Y-%m-%d')}"
                )
            regime_row = regime_predictions[regime_predictions["Date"] == signal_date]
            if regime_row.empty:
                continue
            regime_idx = int(regime_row["Regime_Idx"].iloc[0])
            regime_name = regime_row["Regime_Name"].iloc[0]
            entropy = regime_row["Entropy"].iloc[0]
            current_prices = {}
            for ticker in positions.keys():
                if ticker in asset_data:
                    df = asset_data[ticker]
                    matches = df[df["Date"].dt.date == signal_date.date()]
                    if not matches.empty:
                        current_prices[ticker] = matches.iloc[0]["Close"]
            risky_value = sum(
                (
                    positions.get(ticker, 0) * current_prices.get(ticker, 0)
                    for ticker in positions.keys()
                )
            )
            if day_idx > 0:
                cash += cash * (self.config.RISK_FREE_RATE / 252)
            portfolio_value = cash + risky_value
            days_since_rebal = day_idx - last_rebalance
            should_rebalance = days_since_rebal >= self.config.REBALANCE_FREQUENCY
            if should_rebalance and day_idx >= self.config.OPTIMIZATION_LOOKBACK:
                universe = regime_universes[regime_idx]
                available_assets = [a for a in universe if a in asset_data]
                if len(available_assets) < 3:
                    portfolio_history.append(
                        {
                            "Date": signal_date,
                            "Value": portfolio_value,
                            "Cash": cash,
                            "Risky": risky_value,
                            "Regime": regime_name,
                            "Entropy": entropy,
                        }
                    )
                    continue
                hist_start = max(0, day_idx - self.config.OPTIMIZATION_LOOKBACK)
                hist_dates = trading_dates.iloc[hist_start:day_idx]
                returns_dict = {}
                for asset in available_assets:
                    if asset not in asset_data:
                        continue
                    df = asset_data[asset]
                    asset_returns = []
                    for date in hist_dates:
                        matches = df[df["Date"].dt.date == date.date()]
                        if not matches.empty:
                            close_price = matches.iloc[0]["Close"]
                            asset_returns.append(close_price)
                        else:
                            asset_returns.append(np.nan)
                    if len(asset_returns) > 0:
                        returns_series = pd.Series(asset_returns).pct_change()
                        if returns_series.notna().sum() / len(returns_series) >= 0.8:
                            returns_dict[asset] = returns_series.values
                if len(returns_dict) < 3:
                    portfolio_history.append(
                        {
                            "Date": signal_date,
                            "Value": portfolio_value,
                            "Cash": cash,
                            "Risky": risky_value,
                            "Regime": regime_name,
                            "Entropy": entropy,
                        }
                    )
                    continue
                returns_matrix = pd.DataFrame(returns_dict, index=hist_dates)
                returns_matrix = returns_matrix.dropna(how="all")
                target_weights = self.optimizer.optimize_by_regime(regime_idx, returns_matrix)
                if not target_weights:
                    portfolio_history.append(
                        {
                            "Date": signal_date,
                            "Value": portfolio_value,
                            "Cash": cash,
                            "Risky": risky_value,
                            "Regime": regime_name,
                            "Entropy": entropy,
                        }
                    )
                    continue
                next_idx = day_idx + 1
                if next_idx >= len(trading_dates):
                    portfolio_history.append(
                        {
                            "Date": signal_date,
                            "Value": portfolio_value,
                            "Cash": cash,
                            "Risky": risky_value,
                            "Regime": regime_name,
                            "Entropy": entropy,
                        }
                    )
                    continue
                exec_date = trading_dates.iloc[next_idx]
                exec_prices = {}
                for ticker in target_weights.keys():
                    if ticker in asset_data:
                        df = asset_data[ticker]
                        matches = df[df["Date"].dt.date == exec_date.date()]
                        if not matches.empty:
                            exec_prices[ticker] = matches.iloc[0]["Open"]
                if not exec_prices:
                    portfolio_history.append(
                        {
                            "Date": signal_date,
                            "Value": portfolio_value,
                            "Cash": cash,
                            "Risky": risky_value,
                            "Regime": regime_name,
                            "Entropy": entropy,
                        }
                    )
                    continue
                trade_costs = 0.0
                for ticker, weight in target_weights.items():
                    if ticker in exec_prices:
                        target_value = portfolio_value * weight
                        current_value = positions.get(ticker, 0) * exec_prices[ticker]
                        trade_value = target_value - current_value
                        if abs(trade_value) > portfolio_value * 0.01:
                            cost = self.calculate_transaction_costs(trade_value)
                            shares = trade_value / exec_prices[ticker]
                            positions[ticker] = positions.get(ticker, 0) + shares
                            cash -= trade_value + cost
                            trade_costs += cost
                            self.trades.append(
                                {
                                    "Date": exec_date,
                                    "Ticker": ticker,
                                    "Action": "BUY" if trade_value > 0 else "SELL",
                                    "Value": abs(trade_value),
                                    "Price": exec_prices[ticker],
                                    "Cost": cost,
                                    "Regime": regime_name,
                                }
                            )
                for ticker in list(positions.keys()):
                    if ticker not in target_weights and ticker in exec_prices:
                        if positions[ticker] != 0:
                            value = positions[ticker] * exec_prices[ticker]
                            cost = self.calculate_transaction_costs(abs(value))
                            cash += value - cost
                            positions[ticker] = 0
                            trade_costs += cost
                            self.trades.append(
                                {
                                    "Date": exec_date,
                                    "Ticker": ticker,
                                    "Action": "CLOSE",
                                    "Value": abs(value),
                                    "Price": exec_prices[ticker],
                                    "Cost": cost,
                                    "Regime": regime_name,
                                }
                            )
                total_costs += trade_costs
                last_rebalance = day_idx
            portfolio_history.append(
                {
                    "Date": signal_date,
                    "Value": portfolio_value,
                    "Cash": cash,
                    "Risky": risky_value,
                    "Regime": regime_name,
                    "Entropy": entropy,
                    "Regime_Idx": regime_idx,
                }
            )
        print("\n" + "=" * 80)
        print("CALCULATING PERFORMANCE METRICS")
        print("=" * 80)
        portfolio_df = pd.DataFrame(portfolio_history)
        portfolio_df["Return"] = portfolio_df["Value"].pct_change()
        final_value = portfolio_df["Value"].iloc[-1]
        total_return = final_value / initial_capital - 1
        days = len(portfolio_df)
        annual_return = (1 + total_return) ** (252 / days) - 1
        returns = portfolio_df["Return"].dropna()
        volatility = returns.std() * np.sqrt(252)
        sharpe = (annual_return - self.config.RISK_FREE_RATE) / volatility if volatility > 0 else 0
        cumulative = (1 + returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        max_dd = drawdown.min()
        metrics = {
            "initial_capital": initial_capital,
            "final_value": final_value,
            "total_return": total_return,
            "annual_return": annual_return,
            "volatility": volatility,
            "sharpe_ratio": sharpe,
            "max_drawdown": max_dd,
            "total_costs": total_costs,
            "num_trades": len(self.trades),
        }
        print(f"\n{'=' * 80}")
        print("BACKTEST COMPLETE")
        print(f"{'=' * 80}")
        print(f"Final Value: ${final_value:,.0f}")
        print(f"Total Return: {total_return:.1%}")
        print(f"Annual Return: {annual_return:.1%}")
        print(f"Sharpe Ratio: {sharpe:.2f}")
        print(f"Max Drawdown: {max_dd:.1%}")
        print(f"Transaction Costs: ${total_costs:,.0f}")
        print(f"Number of Trades: {len(self.trades)}")
        return {"portfolio_history": portfolio_df, "trade_history": self.trades, "metrics": metrics}


def main():
    """Execute Entropic Volatility Spread Strategy."""
    print("=" * 80)
    print("ENTROPIC VOLATILITY SPREAD STRATEGY")
    print("=" * 80)
    print("\nCONCEPT:")
    print("  Uses entropy of VIX - Yang-Zhang volatility spread")
    print("  High entropy → Market dislocation → Defensive")
    print("  Low entropy → Market alignment → Aggressive")
    config = EntropyStrategyConfig()
    config.__post_init__()
    print(f"\nCONFIGURATION:")
    print(f"  Period: {config.START_DATE} to {config.END_DATE}")
    print(f"  Yang-Zhang Window: {config.YANG_ZHANG_WINDOW} days")
    print(f"  Z-Score Lookback: {config.ZSCORE_LOOKBACK} days")
    print(f"  Refit Frequency: {config.REFIT_FREQUENCY} days")
    print("\n" + "=" * 80)
    print("STEP 1: DATA ACQUISITION")
    print("=" * 80)
    fetcher = DataFetcher()
    buffer_start = (
        datetime.strptime(config.START_DATE, "%Y-%m-%d") - timedelta(days=300)
    ).strftime("%Y-%m-%d")
    print(f"\nFetching SPY data (with buffer from {buffer_start})...")
    spy_data = fetcher.fetch_historical_data("SPY", buffer_start, config.END_DATE)
    print("\nFetching VIX data...")
    vix_data = fetcher.fetch_historical_data("^VIX", buffer_start, config.END_DATE)
    print("\nFetching asset universe...")
    all_assets = list(
        set(
            config.HIGH_ENTROPY_ASSETS
            + config.MEDIUM_ENTROPY_ASSETS
            + config.LOW_ENTROPY_ASSETS
            + [config.BENCHMARK]
        )
    )
    asset_data = {}
    for ticker in all_assets:
        try:
            df = fetcher.fetch_historical_data(ticker, buffer_start, config.END_DATE)
            asset_data[ticker] = df
        except:
            print(f"  ! Skipping {ticker}")
            continue
    print(f"\n✓ Successfully fetched {len(asset_data)} assets")
    print("\n" + "=" * 80)
    print("STEP 2: ENTROPY REGIME DETECTION")
    print("=" * 80)
    detector = WalkForwardEntropyDetector(config)
    vix_series = vix_data.set_index("Date")["Close"]
    features_df = detector.calculate_features_t_minus_1(spy_data, vix_series)
    regime_predictions, fold_list = detector.walk_forward_detect(features_df)
    print("\n" + "=" * 80)
    print("STEP 3: BACKTEST EXECUTION")
    print("=" * 80)
    backtest_start = pd.to_datetime(config.START_DATE)
    backtest_end = pd.to_datetime(config.END_DATE)
    backtest_data = regime_predictions[
        (regime_predictions["Date"] >= backtest_start)
        & (regime_predictions["Date"] <= backtest_end)
    ].copy()
    print(f"\nBacktest data: {len(backtest_data)} days")
    engine = BacktestEngine(config)
    results = engine.run_backtest(backtest_data, asset_data, config.INITIAL_CAPITAL)
    print("\n" + "=" * 80)
    print("STEP 4: GENERATING VISUALIZATIONS")
    print("=" * 80)
    fig, axes = plt.subplots(4, 1, figsize=(16, 14))
    ax1 = axes[0]
    entropy_data = backtest_data.dropna(subset=["Entropy"])
    ax1.plot(
        entropy_data["Date"],
        entropy_data["Entropy"],
        linewidth=1.5,
        color="darkblue",
        label="Entropy",
    )
    if "Entropy_Threshold_High" in entropy_data.columns:
        ax1.plot(
            entropy_data["Date"],
            entropy_data["Entropy_Threshold_High"],
            linestyle="--",
            color="red",
            alpha=0.5,
            label="High Threshold",
        )
        ax1.plot(
            entropy_data["Date"],
            entropy_data["Entropy_Threshold_Low"],
            linestyle="--",
            color="green",
            alpha=0.5,
            label="Low Threshold",
        )
    ax1.set_title("Student-t Entropy of VIX - Yang-Zhang Spread", fontsize=14, fontweight="bold")
    ax1.set_ylabel("Entropy", fontsize=11)
    ax1.legend(loc="upper left")
    ax1.grid(True, alpha=0.3)
    ax2 = axes[1]
    regime_colors = {0: "green", 1: "orange", 2: "red"}
    for regime_idx in [0, 1, 2]:
        mask = backtest_data["Regime_Idx"] == regime_idx
        if mask.sum() > 0:
            ax2.scatter(
                backtest_data.loc[mask, "Date"],
                backtest_data.loc[mask, "Regime_Idx"],
                c=regime_colors[regime_idx],
                alpha=0.6,
                s=20,
                label=detector.regime_names[regime_idx],
            )
    ax2.set_title("Entropy Regime Classification", fontsize=14, fontweight="bold")
    ax2.set_ylabel("Regime", fontsize=11)
    ax2.set_yticks([0, 1, 2])
    ax2.set_yticklabels(["Low\n(Aggressive)", "Medium\n(Balanced)", "High\n(Defensive)"])
    ax2.legend(loc="upper left", fontsize=9)
    ax2.grid(True, alpha=0.3)
    ax3 = axes[2]
    portfolio_df = results["portfolio_history"]
    ax3.plot(
        portfolio_df["Date"],
        portfolio_df["Value"],
        linewidth=2.5,
        color="darkblue",
        label="Strategy",
    )
    if config.BENCHMARK in asset_data:
        bench_df = asset_data[config.BENCHMARK]
        bench_aligned = []
        for date in portfolio_df["Date"]:
            matches = bench_df[bench_df["Date"].dt.date == date.date()]
            if not matches.empty:
                bench_aligned.append({"Date": date, "Close": matches.iloc[0]["Close"]})
        if bench_aligned:
            bench_plot = pd.DataFrame(bench_aligned)
            first_close = bench_plot["Close"].iloc[0]
            bench_plot["Value"] = bench_plot["Close"] / first_close * config.INITIAL_CAPITAL
            ax3.plot(
                bench_plot["Date"],
                bench_plot["Value"],
                linewidth=2,
                color="orange",
                alpha=0.7,
                label=f"{config.BENCHMARK} (Benchmark)",
            )
    ax3.set_title("Strategy Performance", fontsize=14, fontweight="bold")
    ax3.set_ylabel("Portfolio Value ($)", fontsize=11)
    ax3.legend(loc="upper left")
    ax3.grid(True, alpha=0.3)
    ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))
    ax4 = axes[3]
    returns = portfolio_df["Return"].dropna()
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    ax4.fill_between(
        portfolio_df["Date"].iloc[1:], drawdown.values * 100, 0, alpha=0.7, color="red"
    )
    ax4.set_title("Strategy Drawdown", fontsize=14, fontweight="bold")
    ax4.set_xlabel("Date", fontsize=11)
    ax4.set_ylabel("Drawdown (%)", fontsize=11)
    ax4.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print("\n" + "=" * 80)
    print("EXECUTION COMPLETE")
    print("=" * 80)
    return (results, regime_predictions, fold_list)


if __name__ == "__main__":
    results, regimes, folds = main()
